# ScreamingFace quickstart

Six steps: inspect the public Leaderboards, connect a provider, run a Benchmark, read the
Report, publish its Candidate Result, and replay its URL4. The wider interface is covered in
`01_client_tour.ipynb`.

## Before running

From a terminal in `packages/screamingface/`:

```bash
pip install "screamingface[runtime,notebook]"
screamingface prepare draco  # first run only: download pinned Benchmark assets
screamingface up             # start AI Gateway :9105 and Engine :9108
screamingface status
```

Use `screamingface down` when finished. Stack management stays outside the notebook so **Run All**
never starts or stops local services.

In [1]:
!screamingface prepare draco

cd /Users/flowstatic/screamingface/packages/screamingface/../../apps/url4-cloud && uv run --with datasets python -m url4_cloud.benchmarks.draco.prepare --out /tmp/screamingface-benchmark-assets/draco
Uninstalled 1 package in 2ms                                             
Installed 1 package in 1msfrom file:///Users/flowstatic/scre
⠹ multidict==6.7.1                                                              {"cases": 100, "out": "/tmp/screamingface-benchmark-assets/draco"}
cd /Users/flowstatic/screamingface/packages/screamingface/../../apps/url4-cloud && uv run --with datasets python -m url4_cloud.benchmarks.ifeval.prepare --out /tmp/screamingface-benchmark-assets/ifeval
⠙ python-dateutil==2.9.0.post0                                                  {"cases": 541, "patched_keys": [2785], "out": "/tmp/screamingface-benchmark-assets/ifeval", "nltk_data": "/tmp/screamingface-benchmark-assets/ifeval/nltk_data"}
cd /Users/flowstatic/screamingface/packages/screamingface/../../apps/url4-c

In [4]:
!screamingface up

syncing gateway venv (/Users/flowstatic/screamingface/packages/screamingface/../../apps/aigateway)…
Resolved 83 packages in 0.61ms
Audited 81 packages in 0.10ms
migrating gateway database…
Connection: default
  No migrations to apply
syncing scoreboard venv (/Users/flowstatic/screamingface/packages/screamingface/../../apps/scoreboard)…
Resolved 41 packages in 0.58ms
Audited 40 packages in 0.09ms
migrating scoreboard database…
Connection: default
  No migrations to apply
syncing engine venv (/Users/flowstatic/screamingface/packages/screamingface/../../apps/url4-cloud)…
Resolved 139 packages in 1ms
Audited 64 packages in 0.15ms
seeded benchmark draco/smoke
gateway   :9105 starting  (log: /var/folders/qr/1ch81rkj5hs0bj35gxlk68wm0000gn/T//screamingface-stack/aigateway.log)
scoreboard :9106 starting  (log: /var/folders/qr/1ch81rkj5hs0bj35gxlk68wm0000gn/T//screamingface-stack/scoreboard.log)
engine    :9108 starting  (log: /var/folders/qr/1ch81rkj5hs0bj35gxlk68wm0000gn/T//screamingface-stack

In [9]:
!rustup update stable

info: syncing channel updates for 'stable-aarch64-apple-darwin'
info: latest update on 2026-07-16, rust version 1.97.1 (8bab26f4f 2026-07-14)
info: downloading component 'cargo'
info: downloading component 'clippy'
info: downloading component 'rust-docs'
info: downloading component 'rust-std'
info: downloading component 'rustc'
 65.0 MiB /  65.0 MiB (100 %)  47.7 MiB/s in  1s         
info: downloading component 'rustfmt'
info: removing previous version of component 'cargo'
info: removing previous version of component 'clippy'
info: removing previous version of component 'rust-docs'
info: removing previous version of component 'rust-std'
info: removing previous version of component 'rustc'
info: removing previous version of component 'rustfmt'
info: installing component 'cargo'
info: installing component 'clippy'
info: installing component 'rust-docs'
 22.8 MiB /  22.8 MiB (100 %)   2.6 MiB/s in  7s         
info: installing component 'rust-std'
 27.9 MiB /  27.9 MiB (100 %)  20.0 MiB/

In [5]:
!screamingface status

gateway   :9105 UP   (unrecognized: /Library/Frameworks/Python.framework/Versions/3.13/Resources/Python.app/Contents/MacOS/Python .venv/bin/uvicorn aigateway.main:app --port 9105)
scoreboard   :9106 UP   (unrecognized: /Library/Frameworks/Python.framework/Versions/3.13/Resources/Python.app/Contents/MacOS/Python .venv/bin/uvicorn scoreboard.main:app --port 9106)
engine   :9108 UP   (unrecognized: /Library/Frameworks/Python.framework/Versions/3.13/Resources/Python.app/Contents/MacOS/Python .venv/bin/url4-cloud serve --local)


In [9]:
import screamingface as sf 

client = sf.configure(
    engine_url="http://127.0.0.1:9108",
    scoreboard_url="http://127.0.0.1:9106",
)
client

ModuleNotFoundError: No module named 'screamingface'

## 1 · Leaderboards

Leaderboard discovery reads from the Scoreboard and does not require a provider connection.
`screamingface up` registers the local `draco/smoke` development Leaderboard, so discovery,
evaluation, and publication use the same Benchmark id. Both values render as interactive,
brand-system notebook widgets.

In [18]:
leaderboards = sf.leaderboards.list()
leaderboards

Leaderboards(1)

In [35]:
leaderboard = sf.leaderboards.get("draco/lite", top=10)
leaderboard

LeaderboardError: Leaderboard 'draco/lite' is not registered

## 2 · Connect

`sf.connect()` renders the Engine-backed provider panel. A key entered here goes to the SF
Engine for AI Gateway validation and encrypted storage; the notebook never retains it. On a
hosted Engine the panel asks for Cloudflare Access login first.

In [20]:
sf.connect()

PanelWidget(children=(HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efe…

## 3 · Evaluate

`draco/smoke` keeps DRACO's execution structure but reduces it to one pinned Case, one
criterion, and one Judge pass. It makes two paid calls — one Candidate answer, one Judge
grade — so the score is diagnostic and **not comparable** to canonical DRACO.

Running this cell makes those two inexpensive calls. While it runs, the live panel shows
progress, model calls, tokens and cost.

In [21]:
candidate = sf.Model("openrouter/google/gemini-3-flash-preview")

report = sf.evaluate(candidate, benchmark="draco/lite", limit=1)

HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efeff1;--sf-ink:#16181d;-…

## 4 · Report

The Report renders score, pass rate, coverage, cost and tokens, with every Case and the
Judge's per-criterion reasoning underneath. **&darr; report.json** downloads the portable
artifact — the same complete JSON document `report.export()` writes to the notebook's working
directory.

In [22]:
report

Report(benchmark='draco/lite', candidates=['gemini-3-flash-preview'], ok=True)

In [23]:
artifact_path = report.export()
artifact_path

PosixPath('report.json')

## 5 · Publish and retrieve

Publication accepts the evaluated `CandidateResult` directly. It derives the Benchmark id,
compiled URL4, models, accuracy counts, timestamps, and idempotency key from that immutable
result. Publication is independently opt-in so **Run All** never changes the Scoreboard.

The local Scoreboard accepts writes without login. Hosted deployments may require an
edge-verified identity or keep score submission closed.

In [25]:
PUBLISH_RESULT = True

submission = sf.leaderboards.submit(report.candidates.only) if PUBLISH_RESULT else None
submission if submission is not None else ("Set PUBLISH_RESULT = True to publish this result.")

ValueError: Scoreboard submission requires every Case grade to be binary (0 or 1)

In [31]:
published_score = sf.leaderboards.get_score(submission.id) if submission is not None else None
published_score

In [27]:
updated_leaderboard = (
    sf.leaderboards.get(BENCHMARK_ID, top=10) if submission is not None else leaderboard
)
updated_leaderboard

NameError: name 'leaderboard' is not defined

## 6 · Fork or replay the submitted URL4

`published_score.url4` is the raw evaluation expression stored by the Scoreboard. Its
`.to_python()` method returns an editable Model/Fusion and evaluation cell without spending.
Passing the URL4 itself to `sf.evaluate(...)` instead executes that exact, already
Benchmark-linked expression and returns a normal `Report`; do not pass `benchmark=` or `limit=`
again.

Replay is a fresh paid Evaluation and model output may differ, so it has its own opt-in guard.

In [30]:
fork_python = published_score.url4.to_python() if published_score is not None else None
print(fork_python) if fork_python is not None else "Publish a score to generate its Python fork."

'Publish a score to generate its Python fork.'

In [29]:
REPLAY_RESULT = False

replayed_report = (
    sf.evaluate(published_score.url4) if REPLAY_RESULT and published_score is not None else None
)
replayed_report if replayed_report is not None else (
    "Set REPLAY_RESULT = True after publishing to run the stored URL4 again."
)

'Set REPLAY_RESULT = True after publishing to run the stored URL4 again.'